In [1]:
import json
import traceback
from datetime import datetime, timedelta

import boto3

dynamodb = boto3.resource("dynamodb")

table_potcycle_history = dynamodb.Table("PotCycleHistory")

In [2]:
# ==============================================================================
# Validate Request
# ==============================================================================

def validate_special_group_request(body):

    print("=" * 80)
    print("Validating Request")
    print("=" * 80)

    bathup_year = body.get(

        "bathup_year",

        None

    )

    if bathup_year not in [None, ""]:

        bathup_year = int(

            bathup_year

        )

    selected_cathode_blocks = [

        str(x).strip()

        for x in body.get(

            "cathode_block",

            []

        )

    ]

    selected_side_walls = [

        str(x).strip()

        for x in body.get(

            "side_wall",

            []

        )

    ]

    selected_ramming_pastes = [

        str(x).strip()

        for x in body.get(

            "ramming_paste",

            []

        )

    ]

    selected_shell_types = [

        str(x).strip()

        for x in body.get(

            "shell_type",

            []

        )

    ]

    sort_by = body.get(

        "sort_by",

        "CathodeBlock"

    )

    VALID_SORTS = {

        "CathodeBlock",

        "SideWall",

        "RammingPaste",

        "Class"

    }

    if sort_by not in VALID_SORTS:

        raise ValueError(

            f"sort_by must be one of "

            f"{sorted(VALID_SORTS)}"

        )

    #
    # Only one filter allowed
    #

    total_filters = sum(

        [

            bool(selected_cathode_blocks),

            bool(selected_side_walls),

            bool(selected_ramming_pastes),

            bool(selected_shell_types)

        ]

    )

    if total_filters > 1:

        raise ValueError(

            "Only one filter is allowed "

            "(Cathode Block / Side Wall / "

            "Ramming Paste / Shell Type)."

        )

    print(f"Bathup Year      : {bathup_year}")
    print(f"Cathode Blocks   : {selected_cathode_blocks}")
    print(f"Side Walls       : {selected_side_walls}")
    print(f"Ramming Pastes   : {selected_ramming_pastes}")
    print(f"Shell Types      : {selected_shell_types}")
    print(f"Sort By          : {sort_by}")

    return (

        bathup_year,

        selected_cathode_blocks,

        selected_side_walls,

        selected_ramming_pastes,

        selected_shell_types,

        sort_by

    )

In [30]:
body={
    "bathup_year": 2016,
    "cathode_block": [
        "SERS HC-3"
    ],
    "side_wall": [],
    "ramming_paste": [],
    "shell_type": [],
    "sort_by": "CathodeBlock"
}

In [35]:
bathup_year,selected_cathode_blocks,selected_side_walls,selected_ramming_pastes,selected_shell_types,sort_by = validate_special_group_request(body)

Validating Request
Bathup Year      : 2016
Cathode Blocks   : ['SERS HC-3']
Side Walls       : []
Ramming Pastes   : []
Shell Types      : []
Sort By          : CathodeBlock


In [36]:
bathup_year

2016

In [11]:
# # ==============================================================================
# # Load Required Pot Cycles
# # ==============================================================================

# def load_special_group_cycles():

#     print("=" * 80)
#     print("Loading Required Pot Cycles")
#     print("=" * 80)

#     projection = ",".join([

#         "PotNumber",

#         "PotCycle",

#         "CycleOrg",

#         "BathupDate",

#         "CutoutDate",

#         "CycleStatus",

#         "Age",

#         "Class",

#         "CathodeBlock",

#         "SideWall",

#         "RammingPaste"

#     ])

#     response = table_potcycle_history.scan(

#         ProjectionExpression=projection

#     )

#     items = response.get(

#         "Items",

#         []

#     )

#     while "LastEvaluatedKey" in response:

#         response = table_potcycle_history.scan(

#             ProjectionExpression=projection,

#             ExclusiveStartKey=response["LastEvaluatedKey"]

#         )

#         items.extend(

#             response.get(

#                 "Items",

#                 []

#             )

#         )

#     print(

#         f"Records Loaded : {len(items)}"

#     )

#     filtered = []

#     for item in items:

#         try:

#             bathup_date = datetime.strptime(

#                 item["BathupDate"],

#                 "%Y-%m-%d"

#             )

#             #
#             # Calculate Age
#             #

#             cutout = item.get(

#                 "CutoutDate",

#                 ""

#             ).strip()

#             if cutout:

#                 cutout_date = datetime.strptime(

#                     cutout,

#                     "%Y-%m-%d"

#                 )

#                 age = (

#                     cutout_date

#                     -

#                     bathup_date

#                 ).days

#             else:

#                 cutout_date = None

#                 age = (

#                     datetime.today()

#                     -

#                     bathup_date

#                 ).days

#             record = item.copy()

#             record["BathupDate"] = bathup_date

#             record["CutoutDate"] = cutout_date

#             record["Age"] = age

#             filtered.append(

#                 record

#             )

#         except Exception as e:

#             print(

#                 f"Skipping Pot "

#                 f"{item.get('PotNumber')} : {e}"

#             )

#     print(

#         f"Required Records : {len(filtered)}"

#     )

#     return filtered

In [12]:
# ==============================================================================
# Load Required Pot Cycles
# ==============================================================================

def load_special_group_cycles():

    print("=" * 80)
    print("Loading Required Pot Cycles")
    print("=" * 80)

    projection = ",".join([

        "PotNumber",

        "PotCycle",

        "CycleOrg",

        "BathupDate",

        "CutoutDate",

        "CycleStatus",

        "Age",

        "#C",

        "CathodeBlock",

        "SideWall",

        "RammingPaste"

    ])

    response = table_potcycle_history.scan(

        ProjectionExpression=projection,

        ExpressionAttributeNames={

            "#C": "Class"

        }

    )

    items = response.get(

        "Items",

        []

    )

    while "LastEvaluatedKey" in response:

        response = table_potcycle_history.scan(

            ProjectionExpression=projection,

            ExpressionAttributeNames={

                "#C": "Class"

            },

            ExclusiveStartKey=response["LastEvaluatedKey"]

        )

        items.extend(

            response.get(

                "Items",

                []

            )

        )

    print(

        f"Records Loaded : {len(items)}"

    )

    filtered = []

    for item in items:

        try:

            #
            # Bathup Date
            #

            bathup = item.get(

                "BathupDate",

                ""

            ).strip()

            if not bathup:

                continue

            bathup_date = datetime.strptime(

                bathup,

                "%Y-%m-%d"

            )

            #
            # Cutout Date
            #

            cutout = item.get(

                "CutoutDate",

                ""

            ).strip()

            if cutout:

                cutout_date = datetime.strptime(

                    cutout,

                    "%Y-%m-%d"

                )

                age = (

                    cutout_date

                    -

                    bathup_date

                ).days

            else:

                cutout_date = None

                age = (

                    datetime.today()

                    -

                    bathup_date

                ).days

            record = {

                "PotNumber": item.get(

                    "PotNumber"

                ),

                "PotCycle": int(

                    item.get(

                        "PotCycle",

                        0

                    )

                ),

                "CycleOrg": int(

                    item.get(

                        "CycleOrg",

                        0

                    )

                ),

                "BathupDate": bathup_date,

                "CutoutDate": cutout_date,

                "CycleStatus": str(

                    item.get(

                        "CycleStatus",

                        "1"

                    )

                ),

                "Age": age,

                "Class": item.get(

                    "Class"

                ),

                "CathodeBlock": item.get(

                    "CathodeBlock"

                ),

                "SideWall": item.get(

                    "SideWall"

                ),

                "RammingPaste": item.get(

                    "RammingPaste"

                )

            }

            filtered.append(

                record

            )

        except Exception as e:

            print(

                f"Skipping Pot "

                f"{item.get('PotNumber')} : {e}"

            )

    print(

        f"Required Records : {len(filtered)}"

    )

    return filtered

In [4]:
# ==============================================================================
# Merge Pot Cycles
# ==============================================================================

def merge_special_group_cycles(

    cycle_items

):

    print("=" * 80)
    print("Merging Pot Cycles")
    print("=" * 80)

    merged = {}

    for item in cycle_items:

        key = (

            item["PotNumber"],

            int(item["CycleOrg"])

        )

        #
        # First occurrence
        #

        if key not in merged:

            merged[key] = {

                "Class": item.get("Class"),

                "PotNumber": item.get("PotNumber"),

                "CycleOrg": int(

                    item.get(

                        "CycleOrg",

                        0

                    )

                ),

                "BathupDate": item.get("BathupDate"),

                "Age": int(

                    item.get(

                        "Age",

                        0

                    )

                ),

                "CurrentStatus": str(

                    item.get(

                        "CycleStatus",

                        "1"

                    )

                ),

                "CathodeBlock": item.get("CathodeBlock"),

                "SideWall": item.get("SideWall"),

                "RammingPaste": item.get("RammingPaste")

            }

            continue

        #
        # Earliest Bathup
        #

        if (

            item["BathupDate"]

            <

            merged[key]["BathupDate"]

        ):

            merged[key]["BathupDate"] = item["BathupDate"]

        #
        # Sum Age
        #

        merged[key]["Age"] += int(

            item.get(

                "Age",

                0

            )

        )

        #
        # Latest Status
        #

        merged[key]["CurrentStatus"] = str(

            item.get(

                "CycleStatus",

                merged[key]["CurrentStatus"]

            )

        )

    merged_cycles = list(

        merged.values()

    )

    print(

        f"Unique Pot Cycles : "

        f"{len(merged_cycles)}"

    )

    return merged_cycles

In [5]:
# ==============================================================================
# Generate Special Selected Groups Report
# ==============================================================================

def generate_special_group_report(

    merged_cycles,

    bathup_year,

    selected_cathode_blocks,

    selected_side_walls,

    selected_ramming_pastes,

    selected_shell_types,

    sort_by

):

    print("=" * 80)
    print("Generating Special Selected Groups Report")
    print("=" * 80)

    summary = {}

    for cycle in merged_cycles:

        #
        # Bathup Year Filter
        #

        if (

            bathup_year is not None

            and

            cycle["BathupDate"].year != bathup_year

        ):

            continue

        #
        # Characteristic Filters
        #

        if selected_cathode_blocks:

            if (

                cycle.get(

                    "CathodeBlock"

                )

                not in

                selected_cathode_blocks

            ):

                continue

        elif selected_side_walls:

            if (

                cycle.get(

                    "SideWall"

                )

                not in

                selected_side_walls

            ):

                continue

        elif selected_ramming_pastes:

            if (

                cycle.get(

                    "RammingPaste"

                )

                not in

                selected_ramming_pastes

            ):

                continue

        elif selected_shell_types:

            if (

                cycle.get(

                    "Class"

                )

                not in

                selected_shell_types

            ):

                continue

        #
        # Group Key
        #

        key = (

            cycle.get("Class"),

            cycle.get("CathodeBlock"),

            cycle.get("SideWall"),

            cycle.get("RammingPaste")

        )

        if key not in summary:

            summary[key] = {

                "OperCount": 0,

                "FailCount": 0,

                "OperAge": 0,

                "FailAge": 0

            }

        age = cycle["Age"]

        status = str(

            cycle["CurrentStatus"]

        )

        if status == "0":

            summary[key]["OperCount"] += 1

            summary[key]["OperAge"] += age

        else:

            summary[key]["FailCount"] += 1

            summary[key]["FailAge"] += age

    report = []

    for key, value in summary.items():

        oper_count = value["OperCount"]

        fail_count = value["FailCount"]

        oper_age = (

            round(

                value["OperAge"] / oper_count,

                2

            )

            if oper_count

            else None

        )

        fail_age = (

            round(

                value["FailAge"] / fail_count,

                2

            )

            if fail_count

            else None

        )

        (

            class_name,

            cathode,

            sidewall,

            ramming

        ) = key

        report.append(

            {

                "Class": class_name,

                "CathodeBlock": cathode,

                "SideWall": sidewall,

                "RammingPaste": ramming,

                "Total Pots Installed": oper_count + fail_count,

                "Failed Pots": fail_count,

                "Operating Pots": oper_count,

                "Failure Age": fail_age,

                "Operating Age": oper_age

            }

        )

    #
    # Sorting
    #

    sort_map = {

        "CathodeBlock": (

            "CathodeBlock",

            "Class",

            "SideWall",

            "RammingPaste"

        ),

        "SideWall": (

            "SideWall",

            "Class",

            "CathodeBlock",

            "RammingPaste"

        ),

        "RammingPaste": (

            "RammingPaste",

            "Class",

            "CathodeBlock",

            "SideWall"

        ),

        "Class": (

            "Class",

            "CathodeBlock",

            "SideWall",

            "RammingPaste"

        )

    }

    report = sorted(

        report,

        key=lambda x: tuple(

            str(

                x.get(col, "")

            )

            for col in sort_map[sort_by]

        )

    )

    print(

        f"Rows Generated : {len(report)}"

    )

    return report

In [6]:
# ==============================================================================
# Add Totals & Averages
# ==============================================================================

def add_special_group_totals(

    report

):

    print("=" * 80)
    print("Adding Totals")
    print("=" * 80)

    oper_count = 0
    fail_count = 0

    oper_age_sum = 0
    fail_age_sum = 0

    final_report = []

    for row in report:

        final_report.append(

            row

        )

        row_oper = row["Operating Pots"]

        row_fail = row["Failed Pots"]

        oper_count += row_oper

        fail_count += row_fail

        if row["Operating Age"] is not None:

            oper_age_sum += (

                row["Operating Age"]

                *

                row_oper

            )

        if row["Failure Age"] is not None:

            fail_age_sum += (

                row["Failure Age"]

                *

                row_fail

            )

    avg_oper = (

        round(

            oper_age_sum / oper_count,

            2

        )

        if oper_count

        else None

    )

    avg_fail = (

        round(

            fail_age_sum / fail_count,

            2

        )

        if fail_count

        else None

    )

    final_report.append(

        {

            "Class": "Totals & Averages",

            "CathodeBlock": "",

            "SideWall": "",

            "RammingPaste": "",

            "Total Pots Installed": oper_count + fail_count,

            "Failed Pots": fail_count,

            "Operating Pots": oper_count,

            "Failure Age": avg_fail,

            "Operating Age": avg_oper

        }

    )

    print(

        f"Rows After Totals : {len(final_report)}"

    )

    return final_report

In [ ]:
# bathup_year=1990,

# selected_cathode_blocks=["ELCA 4 BDN"],

# selected_side_walls=[],

# selected_ramming_pastes=[],

# selected_shell_types=[],
# sort_by ="CathodeBlock"

In [ ]:
# body={
#     "bathup_year": 1988,
#     "cathode_block": ["ELCA 4 BDN"],
#     "side_wall": [],
#     "ramming_paste": [],
#     "shell_type": [],
#     "sort_by": "CathodeBlock"
# }

In [47]:
body={
    "bathup_year": 1990,
    "cathode_block": [],
    "side_wall": ["SERS EROX-5"],
    "ramming_paste": [],
    "shell_type": [],
    "sort_by": "CathodeBlock"
}

In [48]:
bathup_year,selected_cathode_blocks,selected_side_walls,selected_ramming_pastes,selected_shell_types,sort_by = validate_special_group_request(body)

Validating Request
Bathup Year      : 1990
Cathode Blocks   : []
Side Walls       : ['SERS EROX-5']
Ramming Pastes   : []
Shell Types      : []
Sort By          : CathodeBlock


In [29]:
selected_cathode_blocks

(['ELCA 4 BDN'],)

In [19]:
 cycle_items = load_special_group_cycles()

Loading Required Pot Cycles
Records Loaded : 10368
Required Records : 10367


In [20]:
merged_cycles = merge_special_group_cycles(cycle_items)


Merging Pot Cycles
Unique Pot Cycles : 9435


In [49]:
report = generate_special_group_report(

            merged_cycles=merged_cycles,

            bathup_year=bathup_year,

            selected_cathode_blocks=selected_cathode_blocks,

            selected_side_walls=selected_side_walls,

            selected_ramming_pastes=selected_ramming_pastes,

            selected_shell_types=selected_shell_types,

            sort_by=sort_by

        )

Generating Special Selected Groups Report
Rows Generated : 3


In [50]:
report

[{'Class': 'L-3(30)',
  'CathodeBlock': 'COVA KA-1',
  'SideWall': 'SERS EROX-5',
  'RammingPaste': 'HOT SERS AMC-73',
  'Total Pots Installed': 1,
  'Failed Pots': 1,
  'Operating Pots': 0,
  'Failure Age': 1215.0,
  'Operating Age': None},
 {'Class': 'MK-2(22)',
  'CathodeBlock': 'ELCA 4 BDN',
  'SideWall': 'SERS EROX-5',
  'RammingPaste': 'HOT SERS AMC-73',
  'Total Pots Installed': 1,
  'Failed Pots': 1,
  'Operating Pots': 0,
  'Failure Age': 1677.0,
  'Operating Age': None},
 {'Class': 'L-3(28)',
  'CathodeBlock': 'SERS HC-3',
  'SideWall': 'SERS EROX-5',
  'RammingPaste': 'HOT SERS AMC-73',
  'Total Pots Installed': 22,
  'Failed Pots': 22,
  'Operating Pots': 0,
  'Failure Age': 1657.77,
  'Operating Age': None}]

In [51]:
report = add_special_group_totals(

            report

        )

Adding Totals
Rows After Totals : 4


In [52]:
report

[{'Class': 'L-3(30)',
  'CathodeBlock': 'COVA KA-1',
  'SideWall': 'SERS EROX-5',
  'RammingPaste': 'HOT SERS AMC-73',
  'Total Pots Installed': 1,
  'Failed Pots': 1,
  'Operating Pots': 0,
  'Failure Age': 1215.0,
  'Operating Age': None},
 {'Class': 'MK-2(22)',
  'CathodeBlock': 'ELCA 4 BDN',
  'SideWall': 'SERS EROX-5',
  'RammingPaste': 'HOT SERS AMC-73',
  'Total Pots Installed': 1,
  'Failed Pots': 1,
  'Operating Pots': 0,
  'Failure Age': 1677.0,
  'Operating Age': None},
 {'Class': 'L-3(28)',
  'CathodeBlock': 'SERS HC-3',
  'SideWall': 'SERS EROX-5',
  'RammingPaste': 'HOT SERS AMC-73',
  'Total Pots Installed': 22,
  'Failed Pots': 22,
  'Operating Pots': 0,
  'Failure Age': 1657.77,
  'Operating Age': None},
 {'Class': 'Totals & Averages',
  'CathodeBlock': '',
  'SideWall': '',
  'RammingPaste': '',
  'Total Pots Installed': 24,
  'Failed Pots': 24,
  'Operating Pots': 0,
  'Failure Age': 1640.12,
  'Operating Age': None}]